# Quantización + fine-tuning MLP27K Montmeló para FINN v3

Este notebook hace el flujo siguiente:

1. Carga el checkpoint bueno `mlp_27k.ckpt`.
2. Comprueba el modelo float contra el CSV.
3. Crea la mejor cuantización directa encontrada: `Q_NoOut_w4_a8_act1`.
4. La evalúa contra el CSV.
5. Hace **fine-tuning cuantizado** para intentar recuperar precisión.
6. Exporta el modelo cuantizado ajustado a ONNX FINN.
7. Opcionalmente crea una versión con preprocesado `/255` para entrada `UINT8`.
8. Intenta `build_dataflow` y deja celdas de debug si FINN falla.

Antes de ejecutar, asegúrate de que en la carpeta del notebook está:

```text
mlp_27k.ckpt
```


## 0. Imports

In [1]:
from pathlib import Path
import os
import csv
import math
import random
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)

print("Torch:", torch.__version__)
print("Working dir:", Path.cwd())


Torch: 1.7.1
Working dir: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


## 1. Rutas y dataset

In [2]:
CKPT_PATH = Path("mlp_27k.ckpt")

candidate_dirs = [
    Path("output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic"),
]

DATASET_DIR = None
for d in candidate_dirs:
    if (d / "data.csv").exists():
        DATASET_DIR = d
        break

if not CKPT_PATH.exists():
    raise FileNotFoundError("No encuentro mlp_27k.ckpt en la carpeta del notebook")

if DATASET_DIR is None:
    raise FileNotFoundError("No encuentro el data.csv del dataset")

CSV_PATH = DATASET_DIR / "data.csv"
print("Checkpoint:", CKPT_PATH.resolve())
print("Dataset:", DATASET_DIR)
print("CSV:", CSV_PATH)

VMIN = -0.6
VMAX = 13.0
WMIN = -math.pi
WMAX = math.pi


def denorm_v_01(vnorm):
    return (VMAX - VMIN) * vnorm + VMIN


def denorm_w_01(wnorm):
    return (WMAX - WMIN) * wnorm + WMIN


def norm_v_01(v):
    return (v - VMIN) / (VMAX - VMIN)


def norm_w_01(w):
    return (w - WMIN) / (WMAX - WMIN)


def image_to_flat_01(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    arr = arr / 255.0
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(0)


def image_to_flat_uint8_np(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    return arr.reshape(1, 576).astype(np.float32)


def load_rows(max_rows=None):
    rows = []
    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = DATASET_DIR / row["image_name"]
            if not img_path.exists():
                continue
            rows.append({
                "image_name": row["image_name"],
                "img_path": img_path,
                "v": float(row["v"]),
                "w": float(row["w"]),
            })
            if max_rows is not None and len(rows) >= max_rows:
                break
    return rows

rows_eval = load_rows(2000)
rows_all = load_rows(None)
print("Frames eval:", len(rows_eval))
print("Frames total:", len(rows_all))


Checkpoint: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp_27k.ckpt
Dataset: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic
CSV: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv
Frames eval: 2000
Frames total: 6539


## 2. Modelo float de referencia

In [3]:
class FloatMLP27K(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(576, 32)
        self.fx = nn.Linear(32, 256)
        self.predictor = nn.Linear(256, 2)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fx(x))
        x = self.predictor(x)
        return x


def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "net", "model"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return obj.state_dict()


def clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        nk = k
        changed = True
        while changed:
            changed = False
            for pref in ["model.", "base_model.", "module.", "net."]:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
                    changed = True
        nk = nk.replace("predictor.0.", "predictor.")
        out[nk] = v
    return out


obj = torch.load(str(CKPT_PATH), map_location="cpu")
sd = clean_state_dict(extract_state_dict(obj))

print("Tensores encontrados en checkpoint:")
for k, v in sd.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

float_model = FloatMLP27K()
float_model.load_state_dict(sd, strict=True)
float_model.eval()
print("Modelo float cargado OK")


Tensores encontrados en checkpoint:
fc1.weight (32, 576)
fc1.bias (32,)
fx.weight (256, 32)
fx.bias (256,)
predictor.weight (2, 256)
predictor.bias (2,)
Modelo float cargado OK


## 3. Función de evaluación contra CSV

In [6]:
def eval_pytorch_model(model, rows, name="model", print_first=100):
    model.eval()
    err_v, err_w = [], []
    pred_v, pred_w = [], []

    with torch.no_grad():
        for i, r in enumerate(rows):
            x = image_to_flat_01(r["img_path"])
            y = model(x)
            if hasattr(y, "value"):
                y = y.value
            y = y.view(-1).detach().cpu().numpy()

            vnorm = float(y[0])
            wnorm = float(y[1])
            v_pred = denorm_v_01(vnorm)
            w_pred = denorm_w_01(wnorm)

            v_csv = r["v"]
            w_csv = r["w"]

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if i < print_first:
                raw_v_csv = round(norm_v_01(v_csv) * 255)
                raw_w_csv = round(norm_w_01(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} "
                    f"raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"{name} v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
        "v_min": float(np.min(pred_v)),
        "v_max": float(np.max(pred_v)),
        "w_min": float(np.min(pred_w)),
        "w_max": float(np.max(pred_w)),
    }

    print()
    print(name)
    print("Frames:", res["frames"])
    print("MAE_v =", res["mae_v"])
    print("MAE_w =", res["mae_w"])
    print("v pred min/max/std =", res["v_min"], res["v_max"], res["std_v"])
    print("w pred min/max/std =", res["w_min"], res["w_max"], res["std_w"])

    return res


## 4. Evaluar modelo float

In [9]:
# ============================================================
# Comprobar zona concreta donde cambia v en el CSV
# ============================================================

START_IDX = 50
END_IDX = 75

print("Comparación alrededor del salto de velocidad")
print("=" * 120)

for idx in range(START_IDX - 1, END_IDX):
    r = rows_eval[idx]

    img_name = r["image_name"]
    img_path = r["img_path"]

    v_csv = float(r["v"])
    w_csv = float(r["w"])

    # Float
    with torch.no_grad():
        x = image_to_flat_01(img_path)
        y_float = float_model(x)

        if hasattr(y_float, "value"):
            y_float = y_float.value

    y_float = y_float.view(-1).detach().cpu().numpy()

    v_float = denorm_v_01(float(y_float[0]))
    w_float = denorm_w_01(float(y_float[1]))

    raw_v_float = int(round(float(y_float[0]) * 255.0))
    raw_w_float = int(round(float(y_float[1]) * 255.0))

    # Cuantizada fine-tuned
    with torch.no_grad():
        y_q = qmodel_ft(x)

        if hasattr(y_q, "value"):
            y_q = y_q.value

    y_q = y_q.view(-1).detach().cpu().numpy()

    v_q = denorm_v_01(float(y_q[0]))
    w_q = denorm_w_01(float(y_q[1]))

    raw_v_q = int(round(float(y_q[0]) * 255.0))
    raw_w_q = int(round(float(y_q[1]) * 255.0))

    raw_v_csv = int(round(((v_csv + 0.6) / 13.6) * 255.0))
    raw_w_csv = int(round(((w_csv + math.pi) / (2.0 * math.pi)) * 255.0))

    print(
        f"{img_name:>8s} | "
        f"CSV v={v_csv:7.3f} w={w_csv:7.3f} raw=({raw_v_csv:3d},{raw_w_csv:3d}) | "
        f"FLOAT v={v_float:7.3f} w={w_float:7.3f} raw=({raw_v_float:3d},{raw_w_float:3d}) | "
        f"QFT v={v_q:7.3f} w={w_q:7.3f} raw=({raw_v_q:3d},{raw_w_q:3d})"
    )

Comparación alrededor del salto de velocidad


NameError: name 'qmodel_ft' is not defined

In [7]:
float_res = eval_pytorch_model(float_model, rows_eval, name="FloatMLP27K", print_first=60)
float_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | FloatMLP27K v=4.8432 w=-0.1681 raw=102,121
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
11.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.168

{'name': 'FloatMLP27K',
 'frames': 2000,
 'mae_v': 0.6021009925648568,
 'mae_w': 0.13719837213784833,
 'std_v': 1.2614186193435533,
 'std_w': 0.6229009507532546,
 'v_min': -0.237543985247612,
 'v_max': 6.210170078277588,
 'w_min': -1.5784582724599736,
 'w_max': 1.6049311443956347}

## 5. Modelo cuantizado elegido: pesos 4 bits, activaciones 8 bits

In [ ]:
class QuantMLP27K_W4A8_NoQuantOut(nn.Module):
    def __init__(self):
        super().__init__()

        self.quant_in = qnn.QuantIdentity(
            bit_width=8,
            return_quant_tensor=True,
            min_val=0.0,
            max_val=1.0
        )

        self.fc1 = qnn.QuantLinear(
            576, 32,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu1 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            32, 256,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu2 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            256, 2,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.quant_in(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fx(x)
        x = self.relu2(x)
        x = self.predictor(x)
        if hasattr(x, "value"):
            x = x.value
        return x


def load_float_weights_into_quant(qmodel, sd):
    qsd = qmodel.state_dict()
    loaded = []
    for k, v in sd.items():
        if k in qsd and tuple(qsd[k].shape) == tuple(v.shape):
            qsd[k] = v
            loaded.append(k)
    qmodel.load_state_dict(qsd, strict=False)
    return loaded


qmodel = QuantMLP27K_W4A8_NoQuantOut()
loaded = load_float_weights_into_quant(qmodel, sd)
qmodel.eval()
print("Pesos cargados en cuantizado:")
for k in loaded:
    print(" ", k)
print("Total:", len(loaded))


## 6. Evaluar cuantizado antes de fine-tuning

In [ ]:
q_before_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_before_finetune", print_first=60)
q_before_res


## 7. Dataset y DataLoaders para fine-tuning

In [ ]:
class MontmeloDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        x = image_to_flat_01(r["img_path"]).view(-1)
        y = torch.tensor([norm_v_01(r["v"]), norm_w_01(r["w"])], dtype=torch.float32)
        return x, y


random.seed(0)
rows_train_val = rows_all.copy()
random.shuffle(rows_train_val)

split = int(0.8 * len(rows_train_val))
train_rows = rows_train_val[:split]
val_rows = rows_train_val[split:]

train_ds = MontmeloDataset(train_rows)
val_ds = MontmeloDataset(val_rows)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


## 8. Fine-tuning cuantizado

In [ ]:
device = torch.device("cpu")
qmodel = qmodel.to(device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4, weight_decay=1e-6)

N_EPOCHS = 50
best_val_loss = 1e9
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    qmodel.train()
    train_losses = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = qmodel(x)
        if hasattr(pred, "value"):
            pred = pred.value

        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    qmodel.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            pred = qmodel(x)
            if hasattr(pred, "value"):
                pred = pred.value
            loss = loss_fn(pred, y)
            val_losses.append(float(loss.item()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in qmodel.state_dict().items()}

    print(f"Epoch {epoch:03d}/{N_EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

print("Mejor val_loss:", best_val_loss)
qmodel.load_state_dict(best_state)
qmodel.eval()


## 9. Evaluar cuantizado después del fine-tuning

In [ ]:
q_after_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_finetuned", print_first=60)
q_after_res


## 10. Guardar modelo fine-tuned

In [ ]:
FT_PTH = Path("q_mlp27k_w4a8_finetuned.pth")
torch.save(qmodel.state_dict(), FT_PTH)
print("Guardado:", FT_PTH)
!ls -lh q_mlp27k_w4a8_finetuned.pth


## 11. Exportar a ONNX FINN

In [ ]:
FT_ONNX = Path("mlp27k_w4a8_finetuned.onnx")

bo.export_finn_onnx(
    qmodel,
    input_shape=(1, 576),
    export_path=str(FT_ONNX)
)

print("Exportado:", FT_ONNX)
!ls -lh mlp27k_w4a8_finetuned.onnx


## 12. Tidy-up del ONNX cuantizado

In [ ]:
MODEL_FOR_FINN_01 = Path("model_for_finn_w4a8_finetuned_01.onnx")

model = ModelWrapper(str(FT_ONNX))
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())
model.save(str(MODEL_FOR_FINN_01))

print("Guardado:", MODEL_FOR_FINN_01)
print("Input:", model.graph.input[0].name, model.get_tensor_shape(model.graph.input[0].name), model.get_tensor_datatype(model.graph.input[0].name))
print("Output:", model.graph.output[0].name, model.get_tensor_shape(model.graph.output[0].name))


## 13. Evaluar ONNX exportado contra CSV

In [ ]:
def eval_onnx_01(onnx_path, rows, name="ONNX", print_first=60):
    model = ModelWrapper(str(onnx_path))
    model = model.transform(InferShapes())
    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name

    err_v, err_w = [], []
    pred_v, pred_w = [], []

    for i, r in enumerate(rows):
        x = image_to_flat_01(r["img_path"]).numpy().astype(np.float32)
        y_dict = execute_onnx(model, {input_name: x})
        y = y_dict[output_name].reshape(-1)

        vnorm = float(y[0])
        wnorm = float(y[1])
        v_pred = denorm_v_01(vnorm)
        w_pred = denorm_w_01(wnorm)
        v_csv = r["v"]
        w_csv = r["w"]

        err_v.append(v_pred - v_csv)
        err_w.append(w_pred - w_csv)
        pred_v.append(v_pred)
        pred_w.append(w_pred)

        if i < print_first:
            print(
                f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} | "
                f"{name} v={v_pred:.4f} w={w_pred:.4f}"
            )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
    }
    print(res)
    return res

onnx_res = eval_onnx_01(MODEL_FOR_FINN_01, rows_eval, name="ONNX_W4A8_finetuned", print_first=60)
onnx_res


## 14. Crear versión con preprocesado /255 para entrada UINT8

In [ ]:
# Esta celda crea un ONNX que espera bytes 0-255 y divide internamente entre 255.
# Es útil para que coincida con el testbench C++ que envía bytes de imagen.

try:
    from finn.util.pytorch import ToTensor
    from qonnx.transformation.merge_onnx_models import MergeONNXModels

    MODEL_FOR_FINN_UINT8 = Path("model_for_finn_w4a8_finetuned_uint8.onnx")

    model = ModelWrapper(str(MODEL_FOR_FINN_01))
    model = model.transform(InferShapes())
    global_inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(global_inp_name)
    print("Shape del modelo 0-1:", ishape)

    PREPROC_ONNX = Path("preproc_div255_w4a8.onnx")
    totensor_pyt = ToTensor()
    bo.export_finn_onnx(totensor_pyt, ishape, str(PREPROC_ONNX))

    pre_model = ModelWrapper(str(PREPROC_ONNX))
    merged = model.transform(MergeONNXModels(pre_model))
    global_inp_name = merged.graph.input[0].name
    merged.set_tensor_datatype(global_inp_name, DataType["UINT8"])

    merged = merged.transform(InferShapes())
    merged = merged.transform(FoldConstants())
    merged = merged.transform(GiveUniqueNodeNames())
    merged = merged.transform(GiveReadableTensorNames())
    merged = merged.transform(InferDataTypes())
    merged = merged.transform(RemoveStaticGraphInputs())
    merged = merged.transform(RemoveUnusedTensors())

    merged.save(str(MODEL_FOR_FINN_UINT8))
    shutil.copyfile(MODEL_FOR_FINN_UINT8, "model.onnx")

    print("Modelo UINT8 para FINN guardado en:", MODEL_FOR_FINN_UINT8)
    print("Copiado como model.onnx")
    print("Input:", merged.graph.input[0].name, merged.get_tensor_shape(merged.graph.input[0].name), merged.get_tensor_datatype(merged.graph.input[0].name))

except Exception as e:
    print("No se pudo crear la versión UINT8 con preprocesado.")
    print("Error:", repr(e))
    print("Copio el modelo 0-1 como model.onnx")
    shutil.copyfile(MODEL_FOR_FINN_01, "model.onnx")


## 15. Evaluar model.onnx con entrada UINT8

In [ ]:
# Si la celda anterior creó el preprocesado, model.onnx espera entrada 0-255.
# Si no, puede esperar entrada 0-1. Mira el datatype impreso en la celda anterior.

model = ModelWrapper("model.onnx")
model = model.transform(InferShapes())
input_name = model.graph.input[0].name
output_name = model.graph.output[0].name
print("Input:", input_name, model.get_tensor_shape(input_name), model.get_tensor_datatype(input_name))
print("Output:", output_name, model.get_tensor_shape(output_name))

err_v, err_w = [], []
for i, r in enumerate(rows_eval):
    # probamos con uint8 0-255 porque model.onnx debería incluir /255
    x = image_to_flat_uint8_np(r["img_path"])
    y_dict = execute_onnx(model, {input_name: x})
    y = y_dict[output_name].reshape(-1)
    v_pred = denorm_v_01(float(y[0]))
    w_pred = denorm_w_01(float(y[1]))
    err_v.append(v_pred - r["v"])
    err_w.append(w_pred - r["w"])
    if i < 10:
        print(f"{r['image_name']} | CSV v={r['v']:.4f} w={r['w']:.4f} | model.onnx v={v_pred:.4f} w={w_pred:.4f}")

err_v = np.array(err_v)
err_w = np.array(err_w)
print("MAE_v =", np.mean(np.abs(err_v)))
print("MAE_w =", np.mean(np.abs(err_w)))


## 16. FINN_BUILD_DIR

In [ ]:
os.environ["FINN_BUILD_DIR"] = "/tmp/FINN_build_mlp27k_w4a8_ft"
Path(os.environ["FINN_BUILD_DIR"]).mkdir(parents=True, exist_ok=True)
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])


## 17. Intentar build_dataflow

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

FINAL_OUTPUT_DIR = "output_final_mlp27k_w4a8_ft"

if Path(FINAL_OUTPUT_DIR).exists():
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print("Borrado output anterior:", FINAL_OUTPUT_DIR)

cfg = build.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    stitched_ip_gen_dcp=True,
    split_large_fifos=True,
    synth_clk_period_ns=10.0,
    board=None,
    fpga_part="xc7z020clg400-1",
    default_mem_mode="const",
    shell_flow_type=None,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

print("Lanzando build_dataflow con model.onnx")
print("Output dir:", FINAL_OUTPUT_DIR)

try:
    build.build_dataflow_cfg("model.onnx", cfg)
    print("FINN build terminado")
except Exception as e:
    print("FINN build ha fallado.")
    print("Error:", repr(e))
    print("Mira las celdas de debug siguientes.")
    raise


## 18. Debug si falla FINN

In [ ]:
!tail -120 output_final_mlp27k_w4a8_ft/build_dataflow.log 2>/dev/null || true


In [ ]:
onnx_files = sorted(Path("/tmp/FINN_build_mlp27k_w4a8_ft").rglob("*.onnx"))
print("ONNX intermedios:", len(onnx_files))
for i, f in enumerate(onnx_files):
    print(i, f)

if onnx_files:
    last_model = onnx_files[-1]
    print("\nInspeccionando:", last_model)
    m = ModelWrapper(str(last_model))
    for node in m.graph.node:
        print(node.name, node.op_type)
